In [1]:
import pandas as pd
import re

In [2]:
df_trans=pd.read_excel('Krasnoyarsk_melzdrav_b2026.xlsx',sheet_name='реестр_02')
df_map =pd.read_excel('Соответствие_статей_затрат1.xlsx')
df_stat=pd.read_excel('статусы.xlsx')

In [3]:
df_trans['Сумма.1'].nunique()

664

In [4]:
# Функция преобразования
def convert_pharmacy_name(name):
    if pd.isna(name):
        return name
    name = str(name).strip()
    if name == 'КРАСНОЯРСК':
        return 'АУП КРС'
    if name == 'Крс Аптека 9':
        return 'Крс Аптека 9.1'
    if name == 'Крс Аптека 53.1':
        return 'Крс Аптека 53.2'
    if name == 'Крс Аптека 22':
        return 'Крс Аптека 22.1'
    if name == 'Крс Аптека 13':
        return 'Крс Аптека 13.1'
    if name == 'Крс Аптека 5':
        return 'Крс Аптека 5.1'
    if name == 'Крс Аптека 10':
        return 'Крс Аптека 10.1'
    if name == 'Крс Аптека 7':
        return 'АУП КРС'
    match = re.match(r'Аптека №([\d\.]+)', name)
    if match:
        return f'Крс Аптека {match.group(1)}'
    match = re.match(r'Аптечный пункт №([\d\.]+)', name)
    if match:
        return f'Крс Аптека {match.group(1)}'
    match = re.match(r'Аптека №(\d+)', name)
    if match:
        return f'Крс Аптека {match.group(1)}'
    return name

In [5]:
# Применяем преобразование
df_trans['Бух.подразделение'] = df_trans['Бух.подразделение'].apply(convert_pharmacy_name)
df_trans['подразделение'] = df_trans['подразделение'].apply(convert_pharmacy_name)

In [6]:
# После применения функции удаляем старые названия (без точки)
df_trans = df_trans[~df_trans['Бух.подразделение'].isin([
    'Крс Аптека 5', 'Крс Аптека 9', 'Крс Аптека 22', 'Крс Аптека 53.1','Крс Аптека 13','Крс Аптека 7','Крс Аптека 10','<...>'
])]

In [7]:
# Приводим названия колонок в таблице соответствий
df_map.columns = ['Бух_статья', 'Фин_статья', 'Статус_аптеки']
# Добавляем статус
df_trans = df_trans.merge(df_stat, how='left', left_on='Бух.подразделение', right_on='Наименование')

In [8]:
# Очистка
def clean_article_name(text):
    if pd.isna(text):
        return ''
    text = str(text)
    text = re.sub(r'\([^)]*\)', '', text)
    text = re.sub(r'\s*\d+\.?\d*\s*$', '', text)
    text = re.sub(r'^(Статьи расходов|Статья расходов)\s+', '', text)
    text = text.strip().strip('.')
    return text

df_map['Бух_статья'] = df_map['Бух_статья'].apply(clean_article_name)

In [9]:
df_trans['Сумма.1'].nunique()

664

In [10]:
# Функция поиска
def find_fin_article(row):
    trans_article = str(row['Бух.статья']).lower()
    trans_status = row['Деятельность']
    exact_match = df_map[
        (df_map['Бух_статья'].str.lower() == trans_article) &
        (df_map['Статус_аптеки'] == trans_status)
    ]
    if not exact_match.empty:
        return exact_match.iloc[0]['Фин_статья']
    
    # по ключевым словам
    for keyword, (fin, needed_status) in keyword_map.items():
        if keyword in trans_article:
            # Если статус совпадает или статус не важен
            if needed_status == trans_status or needed_status == 'Работает':
                return fin
    
    return None

In [11]:
keyword_map = {}

for _, row in df_map.iterrows():
    article = row['Бух_статья']
    fin = row['Фин_статья']
    status = row['Статус_аптеки']
    
    # Разбиваем на слова
    words = article.lower().split()
    for word in words:
        if len(word) >= 4 and word not in ['расходов', 'статьи', 'статья', 'услуги']:
            # Сохраняем как кортеж (фин_статья, статус)
            if word not in keyword_map:
                keyword_map[word] = (fin, status)

In [12]:
# Применяем
df_trans['Фин.статья'] = df_trans.apply(find_fin_article, axis=1)

# Статистика
matched = df_trans['Фин.статья'].notna().sum()
total = len(df_trans)
print(f"Найдено соответствий: {matched} из {total} ({matched/total*100:.1f}%)")

Найдено соответствий: 1600 из 29342 (5.5%)


In [14]:
# Заполняем пустые Бух.статьи
df_trans['Бух.статья'] = df_trans['Бух.статья'].fillna('Прочие расходы')

In [15]:
df_trans['Сумма.1'].nunique()

664

In [16]:
# Создаём словарь для нераспознанных статей
manual_mapping = {
    'Профессиональное обучение ЗА и МПМ (аккредитация сотрудников А)': 'Профессиональное обучение ЗА и МПМ (НМО)',
    'Коммунальные услуги (приним.в НУ)': 'Коммунальные услуги',
    'Расходы на проезд для сотрудников (приним. в НУ)': 'Расходы на проезд для сотрудников',
    'Агентское вознаграждение': 'Агентское вознаграждение',
    'Малоценные О.С. (менее 100 тыс.руб.)': 'Ремонт и ТО оборудования: торговое оборудование',
    'Проезд (приним.в НУ)': 'Проезд',
    'Промо мероприятие (промоутеры)': 'Промо мероприятие (промоутеры)',
    'Дератизация, дезинсекция, дезинфекция': 'Дератизация, дезинсекция, дезинфекция ',
    'Представительские расходы (приним.в НУ)': 'Представительские расходы',
    'Амортизация ОС для БУ': 'Амортизация ОС и НМА',
    'Амортизация ППА аренды по ФСБУ 25/2018': 'Амортизация ОС и НМА',
    'Расходы по договорам ГПХ (приним.в НУ)': 'ИР на открытие: Прочие расходы (развитие аптеки)',
    'Расходы на оплату труда': 'Зарплата (оклад+пост. надбавки)',
    'Расходы на оплату больничных листов': 'Зарплата (оклад+пост. надбавки)',
    'Страховые взносы': 'Налоги на ФОТ',
    'Страховые взносы от НС на производстве и ПЗ': 'Налоги на ФОТ',
    'Услуги ИТ- услуги': 'Прочие корректировки МСФО',
    'Юридическое обслуживание': 'Юридические услуги',
    'Услуги проведения инвентаризаций ТЗ и ДС': 'Прочие корректировки МСФО',
    'Услуги казначейства': 'Прочие корректировки МСФО',
    'Услуги бухгалтерского обслуживания': 'Прочие корректировки МСФО',
    'НДС за счет собств.средств (не приним. в НУ)':'НДС не принимаемый',
    'МЗ ООО':'Прочие расходы',
    'Списание ошибок МДЛП':'Прочие расходы',
    'Проценты по ФСБУ Аренда':'% по кредиту на операционную деятельность',
    'Списание КВ, дозакуп':'Списание товара с ИСГ: товары ЦП',
    'СГ ( норматив)':'Списание товара с ИСГ: товары не ЦП',
    'Убыток 2025 года (приним. для НУ)':'Прибыль (убыток) прошлых лет',
    'Подарки сотрудникам (кроме медалей и знаков отличия)':'Прочие корректировки МСФО',
    'Списание задолженности (не приним. для НУ)':'Расходы/Доходы от неоперационной деятельности',
    'Товары в розничной торговле (по покупной стоимости) НДС 10%':'Себестоимость (розница)',
    'Товары в розничной торговле (по покупной стоимости) НДС 20% (22%)':'Себестоимость (розница)',
    'Товары в розничной торговле (по покупной стоимости)  без НДС ':'Себестоимость (розница)',
    'Приобретение товаров и услуг КРСА-0000143006 от 03.11.2025 10:28:51':'НДС не принимаемый',
    'Приобретение товаров и услуг КРСА-0000169558 от 20.12.2025 14:52:39':'НДС не принимаемый',
    'Приобретение товаров и услуг КРСА-0000168575 от 19.12.2025 11:47:34':'НДС не принимаемый',
    'Приобретение товаров и услуг КРСА-0000167338 от 17.12.2025 16:13:19':'НДС не принимаемый',
    'Приобретение товаров и услуг КРСА-0000141657 от 31.10.2025 11:11:17':'НДС не принимаемый',
    'Приобретение товаров и услуг КРСА-0000144340 от 07.11.2025 8:54:38':'НДС не принимаемый',
    'Приобретение товаров и услуг КРСА-0000134985 от 21.10.2025 11:16:53':'НДС не принимаемый',
    'Приобретение товаров и услуг КРСА-0000141926 от 31.10.2025 13:33:47':'НДС не принимаемый',
    'Приобретение товаров и услуг КРСА-0000150560 от 20.11.2025 16:57:42':'НДС не принимаемый',
    'Приобретение товаров и услуг КРСА-0000135107 от 21.10.2025 12:15:51':'НДС не принимаемый',
    'Маркетинговые услуги НДС 22%':'Прочие корректировки МСФО',
    'Себестоимость реализованного товара': 'Себестоимость (розница)',
    'Выручка от реализации':'ВЫРУЧКА РОЗНИЦА',
    'Пакеты с символикой':'Пакеты с символикой',
    'Инкассация':'Инкассация',
    'Услуги связи: абонентская плата за услуги связи':'Услуги связи: абонентская плата за услуги связи',
    'Охрана: пульт':'Охрана: пульт',
    'Услуги связи: интернет':'Услуги связи: абонентская плата за услуги связи',
    'Полиграфия':'Полиграфия',
    'Вывоз мусора': 'Вывоз мусора',
    'Аренда торгового помещения без комм. услуг': 'Аренда торгового помещения',
    'Прочие расходы на БО': 'Прочие расходы на БО',
    'Хозяйственные товары': 'Хозяйственные товары',
    'ДМС': 'ДМС',
    'СБП': 'СБП',
    'Канцелярские расходы': 'Канцелярские расходы',
    'Ремонт помещения: текущий ремонт': 'Ремонт помещения: текущий ремонт',
    'Расходы на медицинский осмотр и медицинские книжки': 'Расходы на медицинский осмотр и медицинские книжки',
    'Расходы на клининг': 'Расходы на клининг',
    'Ремонт и ТО оборудования: компьютеры и оргтехника': 'Ремонт и ТО оборудования: компьютеры и оргтехника',
    'Медицинская одежда': 'Медицинская одежда',
    'Бумага':'Бумага',
    'Призовой фонд':'Призовой фонд',
    'Почтовые и курьерские услуги':'Почтовые и курьерские услуги',
    'Прочие расходы':'Прочие расходы',
    '<...>': 0  
}

In [17]:
# Проверяем итоговый процент
matched = df_trans['Фин.статья'].notna().sum()
total = len(df_trans)
print(f"Найдено соответствий: {matched} из {total} ({matched/total*100:.1f}%)")

Найдено соответствий: 1600 из 29342 (5.5%)


In [18]:
# Заменяем неправильные Бух.статьи на правильные
def fix_buh_article(article):
    if pd.isna(article):
        return article
    article = str(article)
    
    # Товары и себестоимость
    if article in ['_Мед. товар 20%', '_Мед. товар без НДС', '_Мед. товар 10%']:
        return 'Себестоимость реализованного товара'
    
    # Контрагенты и выручка
    if article in ['ФАРМКОМПЛЕКТ ООО Новосибирский филиал', 
                   'МЕЛОДИЯ ЗДОРОВЬЯ АО', 
                   'ЯНДЕКС.ЕДА ООО', 
                   'ИНСТАМАРТ СЕРВИС ООО']:
        return 'Выручка от реализации'
    
    return article

df_trans['Бух.статья'] = df_trans['Бух.статья'].apply(fix_buh_article)

In [ ]:
# Применяем маппинг
df_trans['Фин.статья'] = df_trans['Бух.статья'].map(manual_mapping)

In [21]:
# Проверяем результат
filled = df_trans['Фин.статья'].notna().sum()
total = len(df_trans)
print(f"Заполнено фин.статей: {filled} из {total} ({filled/total*100:.1f}%)")

Заполнено фин.статей: 29199 из 29342 (99.5%)


In [23]:
# Сохраняем
df_trans.to_excel('проводки_с_фин_статьями_ИТОГ11.xlsx', index=False)
print(f"строк: {len(df_trans)}")
print(f"колонок: {len(df_trans.columns)}")

строк: 29342
колонок: 18
